# Fine-tune Surya OCR 2.5 (Colab)

Full fine-tuning of `datalab-to/surya-ocr-2.5` on pages from our generator (branch `main`, tasks `full_ocr` and
`block_ocr`): the vision encoder is frozen, the rest (602M weights) is trained with 8-bit AdamW.

1. *Runtime > Change runtime type > T4 GPU*.
2. Put the batch folders in Google Drive: `MyDrive/surya/batches/epoch_001`, `epoch_002`, ... One folder per epoch,
   each made by `python -m pipeline.run dataset --n 1000 --out epoch_001 --task full_ocr+block_ocr --seed 1` (new seed
   per epoch). Folders can be added while training runs: the notebook waits for the next one (`WAIT_MINUTES`).
3. *Runtime > Run all*. Progress is saved to Drive; after a disconnect, *Run all* again continues.

In [1]:
%pip install -q transformers==5.22.1 bitsandbytes==0.50.2

In [2]:
import difflib, json, os, random, re, time
from pathlib import Path

import bitsandbytes as bnb
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration, TokenizersBackend, get_cosine_schedule_with_warmup

if "google.colab" in str(get_ipython()):
    from google.colab import drive
    drive.mount("/content/drive")

MODEL = "datalab-to/surya-ocr-2.5"
DATA = Path("/content/drive/MyDrive/surya/batches")  # epoch_001/, epoch_002/, ...
OUT = Path("/content/drive/MyDrive/surya/run1")
EPOCHS = 2
LR = 1e-5
GRAD_ACCUM = 8
MAX_TOKENS = 4096      # longer pages are skipped
SAVE_EVERY = 100       # optimizer steps
EVAL_PAGES = 8
EVAL_MAX_TOKENS = 4096  # longest answer generated when measuring
WAIT_MINUTES = 0       # how long to wait for a batch folder that is not there yet
OUT.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device, torch.cuda.get_device_name(0) if device == "cuda" else "")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2.11.0+cu130 cuda Tesla T4


**Load the model.** Two settings matter, both checked against Surya's own server:

- the tokenizer is loaded as Surya stores it (`TokenizersBackend`). transformers otherwise loads it as a Qwen3.5
  tokenizer, which splits text differently (Arabic becomes byte fragments, `\n` gets another id): with that, the
  model answers block images with nothing;
- images are resized within Surya's limits (at least 1792x28, at most 3072x2048 pixels).

In [3]:
model = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float32).to(device)  # a T4 has no bfloat16
processor = AutoProcessor.from_pretrained(MODEL, tokenizer=TokenizersBackend.from_pretrained(MODEL))
processor.image_processor.size = {"shortest_edge": 1792 * 28, "longest_edge": 3072 * 2048}

for part in [model.model.visual.patch_embed, model.model.visual.pos_embed, model.model.visual.blocks]:
    part.requires_grad_(False)  # the vision encoder stays as it is; its merger and the language model train
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"{sum(p.numel() for p in trainable) / 1e6:.0f}M of {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M weights train")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

601M of 689M weights train


**Data.** Each page: the image and its prompt as the user turn, written by the chat template exactly as Surya is
asked, then the answer as the model writes it: the target's own tokens and the end token `<|im_end|>` (id 2).
The answer is not tokenized together with the chat text: Surya's vocabulary has tokens such as `.<` and `><`, which
would join the target's last character with the `<` of `<|im_end|>` and turn the end token into plain text.
Only the answer counts in the loss (`labels` -100 for the prompt), and in it not the words of picture descriptions
(`<img alt="Photo: ..."/><p class="img-desc">...</p>`): ours are one stock sentence per kind of picture, while Surya
describes each picture; the tags around them are learned.

In [4]:
def batch(epoch):
    folder = DATA / f"epoch_{epoch:03d}"
    deadline = time.time() + WAIT_MINUTES * 60
    while not (folder / "dataset.json").exists():  # the dataset command writes it last
        if time.time() > deadline:
            raise FileNotFoundError(f"{folder} is not there (or not finished)")
        time.sleep(30)
    rows = [json.loads(line) for line in (folder / "manifest.jsonl").read_text(encoding="utf-8").splitlines()]
    return [dict(row, folder=folder) for row in rows if row["task"] in ("full_ocr", "block_ocr")]


DESCRIPTION = re.compile(r'<img alt="Photo: ([^"]*)"/><p class="img-desc">([^<]*)</p>')


def inputs_for(row, with_answer):
    image = Image.open(row["folder"] / row["images"]).convert("RGB")
    prompt = (row["folder"] / row["prompts"]).read_text(encoding="utf-8")
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True,
                                           return_tensors="pt")
    if with_answer:
        target = (row["folder"] / row["targets"]).read_text(encoding="utf-8")
        answer = processor.tokenizer(target, add_special_tokens=False, return_offsets_mapping=True)
        spans = [m.span(g) for m in DESCRIPTION.finditer(target) for g in (1, 2)]
        ids = answer["input_ids"] + [processor.tokenizer.eos_token_id]
        # a token at least half inside a picture description: not in the loss
        labels = [-100 if 2 * sum(max(0, min(e, b) - max(s, a)) for a, b in spans) >= e - s > 0 else t
                  for t, (s, e) in zip(answer["input_ids"], answer["offset_mapping"])]
        added = torch.tensor([ids])
        inputs["input_ids"] = torch.cat([inputs["input_ids"], added], 1)
        inputs["attention_mask"] = torch.cat([inputs["attention_mask"], torch.ones_like(added)], 1)
        inputs["mm_token_type_ids"] = torch.cat([inputs["mm_token_type_ids"], torch.zeros_like(added)], 1)
        # labels for the last prompt position and the answer (the model shifts them by one)
        inputs["labels"] = torch.tensor([[-100] + labels + [processor.tokenizer.eos_token_id]])
    return inputs.to(device)


val = [row for row in batch(1) if row["split"] == "val"][:EVAL_PAGES] or batch(1)[:EVAL_PAGES]  # no val pages: the first ones


def evaluate(name):
    model.eval()
    scores = []
    for row in val:
        inputs = inputs_for(row, with_answer=False)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=EVAL_MAX_TOKENS, do_sample=False, use_cache=True)
        answer = processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        target = (row["folder"] / row["targets"]).read_text(encoding="utf-8")
        no_words = lambda text: DESCRIPTION.sub('<img alt="Photo: "/><p class="img-desc"></p>', text)  # not trained
        scores.append(difflib.SequenceMatcher(None, no_words(answer), no_words(target)).ratio())
    model.train()
    print(f"{name}: answers {sum(scores) / len(scores):.1%} similar to the targets on {len(scores)} pages")
    return scores


if not (OUT / "before.json").exists():
    (OUT / "before.json").write_text(json.dumps(evaluate("before training")))

**Train.** One epoch per batch folder; progress is saved every `SAVE_EVERY` steps and after every epoch.
After every epoch the validation pages are read again and the similarity is saved to `history.json`.

In [5]:
steps = EPOCHS * -(-sum(row["split"] == "train" for row in batch(1)) // GRAD_ACCUM)
optimizer = bnb.optim.AdamW8bit(trainable, lr=LR, weight_decay=0.0)
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, steps // 20), steps)
state = {"epoch": 1, "page": 0, "step": 0}
if (OUT / "state.pt").exists():
    saved = torch.load(OUT / "state.pt", map_location=device, weights_only=False)
    model.load_state_dict(saved["model"], strict=False)
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    state = saved["state"]
    del saved
    print("continuing from", state)


def save():
    trained = {n: p for n, p in model.named_parameters() if p.requires_grad}
    torch.save({"model": trained, "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "state": state}, OUT / "state.tmp")
    os.replace(OUT / "state.tmp", OUT / "state.pt")


model.train()
while state["epoch"] <= EPOCHS:
    rows = [row for row in batch(state["epoch"]) if row["split"] == "train"]
    random.Random(state["epoch"]).shuffle(rows)
    start, first, losses = time.time(), state["page"], []
    for i in range(first, len(rows)):
        inputs = inputs_for(rows[i], with_answer=True)
        if inputs["input_ids"].shape[1] <= MAX_TOKENS:
            loss = model(**inputs, logits_to_keep=inputs["labels"].shape[1]).loss  # logits only where labels are
            (loss / GRAD_ACCUM).backward()
            losses.append(loss.item())
        if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(rows):
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            state.update(page=i + 1, step=state["step"] + 1)
            gpu = f", GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB" if device == "cuda" else ""
            print(f"epoch {state['epoch']} step {state['step']}/{steps}: loss {sum(losses) / max(1, len(losses)):.4f}, "
                  f"{(time.time() - start) / (i + 1 - first):.1f} s/page{gpu}")
            losses = []
            if state["step"] % SAVE_EVERY == 0:
                save()
    state.update(epoch=state["epoch"] + 1, page=0)
    save()
    scores = evaluate(f"after epoch {state['epoch'] - 1}")  # the same validation pages every time
    history = json.loads((OUT / "history.json").read_text()) if (OUT / "history.json").exists() else {}
    history[state["epoch"] - 1] = sum(scores) / len(scores)
    (OUT / "history.json").write_text(json.dumps(history, indent=1))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


epoch 1 step 1/238: loss 0.3363, 5.3 s/page, GPU 8.5 GB
epoch 1 step 2/238: loss 1.2214, 6.1 s/page, GPU 9.9 GB
epoch 1 step 3/238: loss 0.5225, 5.1 s/page, GPU 9.9 GB
epoch 1 step 4/238: loss 0.4468, 5.3 s/page, GPU 9.9 GB
epoch 1 step 5/238: loss 0.3017, 4.7 s/page, GPU 9.9 GB
epoch 1 step 6/238: loss 0.2266, 4.9 s/page, GPU 9.9 GB
epoch 1 step 7/238: loss 0.2196, 4.8 s/page, GPU 9.9 GB
epoch 1 step 8/238: loss 0.3179, 5.1 s/page, GPU 9.9 GB
epoch 1 step 9/238: loss 0.1521, 5.1 s/page, GPU 9.9 GB
epoch 1 step 10/238: loss 0.1547, 4.8 s/page, GPU 9.9 GB
epoch 1 step 11/238: loss 0.1678, 4.7 s/page, GPU 9.9 GB
epoch 1 step 12/238: loss 0.2176, 4.6 s/page, GPU 9.9 GB
epoch 1 step 13/238: loss 0.1640, 4.4 s/page, GPU 9.9 GB
epoch 1 step 14/238: loss 0.1477, 4.3 s/page, GPU 9.9 GB
epoch 1 step 15/238: loss 0.0548, 4.2 s/page, GPU 9.9 GB
epoch 1 step 16/238: loss 0.2358, 4.3 s/page, GPU 9.9 GB
epoch 1 step 17/238: loss 0.0192, 4.2 s/page, GPU 9.9 GB
epoch 1 step 18/238: loss 0.0805, 4.1 s/

[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


after epoch 1: answers 89.0% similar to the targets on 8 pages
epoch 2 step 120/238: loss 0.1594, 5.4 s/page, GPU 9.9 GB
epoch 2 step 121/238: loss 0.0519, 3.6 s/page, GPU 9.9 GB
epoch 2 step 122/238: loss 0.0888, 3.6 s/page, GPU 9.9 GB
epoch 2 step 123/238: loss 0.1431, 3.4 s/page, GPU 9.9 GB
epoch 2 step 124/238: loss 0.2722, 3.6 s/page, GPU 9.9 GB
epoch 2 step 125/238: loss 0.1992, 3.3 s/page, GPU 9.9 GB
epoch 2 step 126/238: loss 0.1246, 3.4 s/page, GPU 9.9 GB
epoch 2 step 127/238: loss 0.1746, 3.2 s/page, GPU 9.9 GB
epoch 2 step 128/238: loss 0.1635, 3.4 s/page, GPU 9.9 GB
epoch 2 step 129/238: loss 0.1413, 3.4 s/page, GPU 9.9 GB
epoch 2 step 130/238: loss 0.0899, 3.4 s/page, GPU 9.9 GB
epoch 2 step 131/238: loss 0.0433, 3.3 s/page, GPU 9.9 GB
epoch 2 step 132/238: loss 0.0976, 3.3 s/page, GPU 9.9 GB
epoch 2 step 133/238: loss 0.3209, 3.4 s/page, GPU 9.9 GB
epoch 2 step 134/238: loss 0.1373, 3.5 s/page, GPU 9.9 GB
epoch 2 step 135/238: loss 0.1535, 3.7 s/page, GPU 9.9 GB
epoch 2 s

In [6]:
before = json.loads((OUT / "before.json").read_text())
history = json.loads((OUT / "history.json").read_text())
print(f"before training: {sum(before) / len(before):.1%}")
for epoch, score in history.items():
    print(f"after epoch {epoch}: {score:.1%}")
model.save_pretrained(OUT / "model")
processor.save_pretrained(OUT / "model")
print("model saved to", OUT / "model")

before training: 77.8%
after epoch 1: 89.0%
after epoch 2: 88.6%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model saved to /content/drive/MyDrive/surya/run1/model


In [7]:
%pip install -q rapidfuzz
import gc, html, re
from rapidfuzz.distance import Levenshtein

PAGES_PER_TASK = 20   # held-out pages per task (full_ocr, block_ocr); a full page takes about 1-3 minutes per model on a T4
LAYOUT_PAGES = 5      # pages asked the layout prompt (it was not trained; it must still give clean JSON)
REAL = Path("/content/drive/MyDrive/surya/real")  # optional: real pages, each image with a .txt of the same name holding its text
MAX_NEW = 4096        # longest answer; an answer that reaches it did not stop by itself
PROMPT_FULL = "OCR this image to HTML. Each block is a div with data-label and data-bbox (x0 y0 x1 y1, normalized 0-1000)."

for name in ("optimizer", "scheduler"):  # training is over: free the GPU memory they hold
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
models = {"original": Qwen3_5ForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float32).to(device).eval(),
          "trained": model.eval()}

HEADER = re.compile(r"<\|([A-Za-z-]+)\|><\|(\d+)\|><\|(\d+)\|><\|(\d+)\|><\|(\d+)\|>")
LAYOUT_PROMPT = ('Output the layout of this image as JSON. Each entry is a dict with "label", "bbox", and "count" fields. '
                 'Bbox is x0 y0 x1 y1, normalized 0-1000.')


def ask(m, image, prompt, max_new=MAX_NEW):
    """The model's answer, and whether it ended with the end token (stopped by itself)."""
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True,
                                           return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new, do_sample=False, use_cache=True)
    new = out[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new, skip_special_tokens=True), bool((new == processor.tokenizer.eos_token_id).any())


def text_of(answer):
    """The page's text only: no labels, boxes, tags or picture descriptions."""
    answer = DESCRIPTION.sub(" ", answer)
    answer = re.sub(r"<\|[^|>]*\|>|<[^>]+>", " ", answer)
    return " ".join(html.unescape(answer).split())


def valid(task, answer):
    """Surya's answer form: full page = blocks with label and box; block = content only; no </div>, no new lines."""
    if "\n" in answer or "</div>" in answer or re.search(r"</t[dhr]>", answer):
        return False
    return bool(HEADER.match(answer)) if task == "full_ocr" else not HEADER.search(answer)


def same_labels(a, b):
    """Of the blocks in answer a, the share that answer b has in the same place (box overlap >= 0.5) with the same label."""
    A = [(m.group(1), [int(v) for v in m.groups()[1:]]) for m in HEADER.finditer(a)]
    B = [(m.group(1), [int(v) for v in m.groups()[1:]]) for m in HEADER.finditer(b)]
    def iou(p, q):
        w, h = min(p[2], q[2]) - max(p[0], q[0]), min(p[3], q[3]) - max(p[1], q[1])
        inter = max(0, w) * max(0, h)
        return inter / max(1, (p[2] - p[0]) * (p[3] - p[1]) + (q[2] - q[0]) * (q[3] - q[1]) - inter)
    hits = sum(any(la == lb and iou(ba, bb) >= 0.5 for lb, bb in B) for la, ba in A)
    return hits / max(1, len(A))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [8]:
held_out = [row for epoch in range(1, EPOCHS + 1) for row in batch(epoch) if row["split"] == "val"]
pages = [row for task in ("full_ocr", "block_ocr") for row in [r for r in held_out if r["task"] == task][:PAGES_PER_TASK]]
real = [(p, p.with_suffix(".txt")) for p in sorted(REAL.glob("*")) if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        and p.with_suffix(".txt").exists()] if REAL.exists() else []
jobs = ([(f"{row['folder'].name}/{row['id']}", row["task"], row) for row in pages]
        + [(f"layout/{row['folder'].name}/{row['id']}", "layout", row) for row in [r for r in held_out if r["task"] == "full_ocr"][:LAYOUT_PAGES]]
        + [(f"real/{image.name}", "real", image) for image, _ in real])
saved = OUT / "eval_answers.json"
answers = json.loads(saved.read_text()) if saved.exists() else {}
print(len(jobs), "pages to ask each model:", len(pages), "held-out,", LAYOUT_PAGES, "layout,", len(real), "real")
for key, task, item in jobs:
    for name, m in models.items():
        if f"{name}|{key}" in answers:
            continue
        if task == "real":
            image, prompt = Image.open(item).convert("RGB"), PROMPT_FULL
        else:
            image = Image.open(item["folder"] / item["images"]).convert("RGB")
            prompt = LAYOUT_PROMPT if task == "layout" else (item["folder"] / item["prompts"]).read_text(encoding="utf-8")
        start = time.time()
        answer, stopped = ask(m, image, prompt, 3072 if task == "layout" else MAX_NEW)
        answers[f"{name}|{key}"] = {"answer": answer, "stopped": stopped, "seconds": round(time.time() - start, 1)}
        saved.write_text(json.dumps(answers, ensure_ascii=False))
        print(f"{name:8s} {key}: {time.time() - start:.0f} s{'' if stopped else ', did not stop'}")

45 pages to ask each model: 40 held-out, 5 layout, 0 real
original epoch_001/0000103: 62 s
trained  epoch_001/0000103: 51 s
original epoch_001/0000106: 25 s
trained  epoch_001/0000106: 27 s
original epoch_001/0000121: 15 s
trained  epoch_001/0000121: 16 s
original epoch_001/0000165: 24 s
trained  epoch_001/0000165: 24 s
original epoch_001/0000167: 89 s
trained  epoch_001/0000167: 86 s
original epoch_001/0000182: 26 s
trained  epoch_001/0000182: 29 s
original epoch_001/0000180: 37 s
trained  epoch_001/0000180: 57 s
original epoch_001/0000194: 35 s
trained  epoch_001/0000194: 39 s
original epoch_001/0000202: 29 s
trained  epoch_001/0000202: 34 s
original epoch_001/0000282: 112 s
trained  epoch_001/0000282: 121 s
original epoch_001/0000335: 21 s
trained  epoch_001/0000335: 23 s
original epoch_001/0000362: 24 s
trained  epoch_001/0000362: 28 s
original epoch_001/0000442: 6 s
trained  epoch_001/0000442: 6 s
original epoch_001/0000605: 43 s
trained  epoch_001/0000605: 58 s
original epoch_001

In [3]:
from collections import Counter


def word_errors(answer_text, truth_text):
    """Words missed plus extra words, ignoring their order (so a different block order or split is not an error)."""
    a, t = Counter(answer_text.split()), Counter(truth_text.split())
    return sum((t - a).values()) + sum((a - t).values()), sum(t.values())


def clean_json(answer):
    try:
        return isinstance(json.loads(answer.strip()), list)
    except ValueError:
        return False


groups = {}
for key, task, item in jobs:
    o, t = answers[f"original|{key}"], answers[f"trained|{key}"]
    if task == "layout":
        g = groups.setdefault(("layout", "-"), {"n": 0, "json_o": 0, "json_t": 0})
        g["n"] += 1; g["json_o"] += clean_json(o["answer"]); g["json_t"] += clean_json(t["answer"])
        continue
    if task == "real":
        task, lang, truth = "full_ocr", "real", " ".join(item.with_suffix(".txt").read_text(encoding="utf-8").split())
    else:
        lang, truth = item["lang"], text_of((item["folder"] / item["targets"]).read_text(encoding="utf-8"))
    g = groups.setdefault((task, lang), Counter())
    g["n"] += 1
    for side, a in (("o", o), ("t", t)):
        text = text_of(a["answer"])
        g["chars_" + side] += Levenshtein.distance(text, truth)
        e, n = word_errors(text, truth)
        g["words_" + side] += e
        g["stop_" + side] += a["stopped"]
        g["ok_" + side] += valid(task, a["answer"])
    g["chars"] += len(truth); g["words"] += n
    g["same"] += same_labels(o["answer"], t["answer"]) if task == "full_ocr" else 1

print("original -> trained (lower errors are better)")
print("  words wrong: words missed or extra, order ignored (the main score for full pages)")
print("  chars wrong: character errors in reading order (block order counts too)", end="\n\n")
for (task, lang), g in sorted(groups.items()):
    n = g["n"]
    if task == "layout":
        print(f"layout prompt ({n} pages): clean JSON {g['json_o']}/{n} -> {g['json_t']}/{n}")
        continue
    print(f"{task:9s} {lang:4s} ({n:2d} pages)  words wrong {g['words_o'] / g['words']:6.1%} -> {g['words_t'] / g['words']:6.1%}"
          f"   chars wrong {g['chars_o'] / g['chars']:6.1%} -> {g['chars_t'] / g['chars']:6.1%}"
          f"   stopped {g['stop_o']}/{n} -> {g['stop_t']}/{n}   right form {g['ok_o']}/{n} -> {g['ok_t']}/{n}"
          + (f"   same blocks as the original {g['same'] / n:.0%}" if task == "full_ocr" else ""))

NameError: name 'jobs' is not defined

In [2]:
# ===== GRADIO: upload an image and see the trained model's answer (and the original's, to compare) =====
import gc, html, importlib.util, json, re, subprocess, sys, threading, time
from pathlib import Path

if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
import gradio as gr
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration, TextIteratorStreamer, TokenizersBackend

MODEL = "datalab-to/surya-ocr-2.5"
TRAINED = Path("/content/drive/MyDrive/surya/run1/model")  # where the training notebook saved the model
device = "cuda" if torch.cuda.is_available() else "cpu"
PROMPTS = {  # exactly Surya's prompts
    "full_ocr": "OCR this image to HTML. Each block is a div with data-label and data-bbox (x0 y0 x1 y1, normalized 0-1000).",
    "block_ocr": "OCR this block image to HTML.",
    "layout": 'Output the layout of this image as JSON. Each entry is a dict with "label", "bbox", and "count" fields. Bbox is x0 y0 x1 y1, normalized 0-1000.',
    "table": 'Output the table rows then columns as JSON. Each entry is a dict with "label" ("Row" or "Col") and "bbox" (x0 y0 x1 y1, normalized 0-1000).',
}
HEADER = re.compile(r"<\|([A-Za-z-]+)\|><\|(\d+)\|><\|(\d+)\|><\|(\d+)\|><\|(\d+)\|>")
PALETTE = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#42d4f4", "#f032e6", "#9a6324", "#808000", "#000075"]

# training is over: free the optimizer's GPU memory; reuse the models already loaded in this session
for name in ("optimizer", "scheduler"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
if "processor" not in globals():  # loaded as in the training notebook (TokenizersBackend: Arabic is read right)
    processor = AutoProcessor.from_pretrained(MODEL, tokenizer=TokenizersBackend.from_pretrained(MODEL))
    processor.image_processor.size = {"shortest_edge": 1792 * 28, "longest_edge": 3072 * 2048}
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")


def load_trained():
    if (TRAINED / "config.json").exists():  # saved by the last cell of the training notebook
        return Qwen3_5ForConditionalGeneration.from_pretrained(str(TRAINED), dtype=torch.float32)
    state = TRAINED.parent / "state.pt"     # otherwise: the original model plus the trained weights of the last save
    if state.exists():
        print("no saved model folder; using", state)
        m = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float32)
        saved = torch.load(state, map_location="cpu", weights_only=False)
        print("trained up to", saved["state"])
        m.load_state_dict(saved["model"], strict=False)
        del saved
        return m
    run = TRAINED.parent
    raise FileNotFoundError(f"No trained model in {run}. It holds: "
                            f"{sorted(p.name for p in run.iterdir()) if run.exists() else 'nothing (folder missing)'}. "
                            "Set TRAINED to the folder that has config.json.")


if "models" not in globals():
    models = {}
if "trained" not in models:
    models["trained"] = (model if "model" in globals() else load_trained().to(device)).eval()
def get_model(name):
    if name not in models:  # the original Surya, loaded the first time it is asked for
        models[name] = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL, dtype=torch.float32).to(device).eval()
    return models[name]


def color(label):
    return PALETTE[sum(map(ord, label)) % len(PALETTE)]


def boxes_of(answer, task):
    """[(label, [x0, y0, x1, y1] in 0-1000)] from a full page answer, or from layout/table JSON."""
    if task == "full_ocr":
        return [(m.group(1), [int(v) for v in m.groups()[1:]]) for m in HEADER.finditer(answer)]
    if task in ("layout", "table"):  # read entry by entry, so a cut-off answer still shows its boxes
        entries = re.findall(r'"label":\s*"([^"]+)",\s*"bbox":\s*"(\d+) (\d+) (\d+) (\d+)"', as_json(answer))
        return [(e[0], [int(v) for v in e[1:]]) for e in entries]
    return []


def as_json(answer):
    """Surya sometimes writes box numbers inside layout JSON as its coordinate tokens (<|434|>): back to plain numbers."""
    return re.sub(r"\s*<\|(\d+)\|>", r" \1", answer).strip()


def draw(image, boxes):
    image = image.copy()
    W, H = image.size
    pen = ImageDraw.Draw(image)
    try:
        font = ImageFont.load_default(size=max(12, W // 70))
    except TypeError:  # older Pillow
        font = ImageFont.load_default()
    for i, (label, (x0, y0, x1, y1)) in enumerate(boxes):
        box = [x0 * W / 1000, y0 * H / 1000, x1 * W / 1000, y1 * H / 1000]
        pen.rectangle(box, outline=color(label), width=max(2, W // 400))
        tag = f"{i + 1} {label}"
        left, top, right, bottom = pen.textbbox((box[0], box[1]), tag, font=font)
        pen.rectangle([left, top - (bottom - top), right + 4, top], fill=color(label))
        pen.text((box[0] + 2, top - (bottom - top)), tag, fill="white", font=font)
    return image


def render(answer, task):
    """The answer as a readable page: each block with its label, Arabic right to left."""
    if task == "full_ocr":
        found = list(HEADER.finditer(answer))
        parts = []
        for i, m in enumerate(found):
            content = answer[m.end(): found[i + 1].start() if i + 1 < len(found) else len(answer)]
            parts.append(f'<div style="border-left:4px solid {color(m.group(1))};margin:6px 0;padding:4px 10px">'
                         f'<small style="color:{color(m.group(1))}"><b>{i + 1} {m.group(1)}</b> [{" ".join(m.groups()[1:])}]</small>'
                         f'<div dir="auto">{content}</div></div>')
        return "".join(parts) or f'<div dir="auto">{answer}</div>'
    if task == "block_ocr":
        return f'<div dir="auto" style="padding:4px 10px">{answer}</div>'
    try:
        return f"<pre style='white-space:pre-wrap'>{html.escape(json.dumps(json.loads(as_json(answer)), ensure_ascii=False, indent=1))}</pre>"
    except ValueError:
        return f"<p><b>Not valid JSON</b> (cut off?)</p><pre style='white-space:pre-wrap'>{html.escape(as_json(answer))}</pre>"


def ask(name, image, task, max_new):
    """Yields the answer as it is written, then the answer with (stopped, tokens, seconds)."""
    m = get_model(name)
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPTS[task]}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True,
                                           return_tensors="pt").to(device)
    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
    result = {}

    def work():
        try:
            with torch.no_grad():
                result["out"] = m.generate(**inputs, max_new_tokens=int(max_new), do_sample=False, use_cache=True,
                                           streamer=streamer)
        except Exception as error:  # e.g. out of GPU memory: end the stream so the page does not wait forever
            result["error"] = error
            streamer.end()

    start, answer = time.time(), ""
    thread = threading.Thread(target=work)
    thread.start()
    for piece in streamer:
        answer += piece
        yield answer, None
    thread.join()
    if "error" in result:
        torch.cuda.empty_cache()
        raise gr.Error(f"{name} model failed: {result['error']}")
    new = result["out"][0, inputs["input_ids"].shape[1]:]
    stopped = bool((new == processor.tokenizer.eos_token_id).any())
    yield answer, (stopped, len(new), time.time() - start)


def status(info):
    stopped, tokens, seconds = info
    return f"{tokens} tokens in {seconds:.0f} s" + ("" if stopped else ", **did not stop by itself** (raise max new tokens?)")


def run(image, task, compare, max_new):
    if image is None:
        raise gr.Error("Upload an image first.")
    image = image.convert("RGB")
    out = [None, "", "", "*writing...*", None, "", "", ""]
    for slot, name in ([(0, "trained"), (4, "original")] if compare else [(0, "trained")]):
        if name == "original":
            out[7] = "*loading the original model...*" if name not in models else "*writing...*"
            yield out
        for answer, info in ask(name, image, task, max_new):
            out[slot + 1] = answer
            if info is None:
                out[slot + 3] = f"*writing... {len(answer)} characters*"
            else:
                out[slot] = draw(image, boxes_of(answer, task))
                out[slot + 2] = render(answer, task)
                out[slot + 3] = status(info)
            yield out
        torch.cuda.empty_cache()


with gr.Blocks(title="Surya OCR: trained vs original") as demo:
    gr.Markdown("## Surya OCR, trained model\nUpload a page (or a cut-out block for *block_ocr*). "
                "Tick *compare* to also ask the original Surya, below.")
    with gr.Row():
        image_in = gr.Image(type="pil", label="Image", height=420)
        with gr.Column():
            task_in = gr.Radio(list(PROMPTS), value="full_ocr", label="Task (Surya's prompt)")
            compare_in = gr.Checkbox(value=True, label="compare with the original Surya")
            max_in = gr.Slider(256, 8192, value=4096, step=256, label="max new tokens")
            go = gr.Button("Read", variant="primary")
    outputs = []
    for title in ("Trained", "Original"):
        gr.Markdown(f"### {title}")
        info = gr.Markdown()
        with gr.Row():
            boxes = gr.Image(label="boxes", height=600)
            page = gr.HTML()
        raw = gr.Textbox(label="raw answer", lines=6, max_lines=20)
        outputs += [boxes, raw, page, info]
    go.click(run, [image_in, task_in, compare_in, max_in], outputs)

demo.queue().launch(share=False, debug=False)  # share=True also gives a public link (valid 72 hours)

Mounted at /content/drive


Loading weights:   0%|          | 0/473 [00:01<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [4]:
# ===== RESULTS ONLY: scores from the saved answers (no GPU, no model; works in a fresh runtime) =====
%pip install -q rapidfuzz
import html, json, re
from collections import Counter
from pathlib import Path
from rapidfuzz.distance import Levenshtein

DATA = Path("/content/drive/MyDrive/surya/batches")
OUT = Path("/content/drive/MyDrive/surya/run1")
REAL = Path("/content/drive/MyDrive/surya/real")
EPOCHS, PAGES_PER_TASK, LAYOUT_PAGES = 2, 20, 5   # as in the evaluation cells
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DESCRIPTION = re.compile(r'<img alt="Photo: ([^"]*)"/><p class="img-desc">([^<]*)</p>')
HEADER = re.compile(r"<\|([A-Za-z-]+)\|><\|(\d+)\|><\|(\d+)\|><\|(\d+)\|><\|(\d+)\|>")


def batch(epoch):
    folder = DATA / f"epoch_{epoch:03d}"
    rows = [json.loads(line) for line in (folder / "manifest.jsonl").read_text(encoding="utf-8").splitlines()]
    return [dict(row, folder=folder) for row in rows if row["task"] in ("full_ocr", "block_ocr")]


def text_of(answer):
    answer = DESCRIPTION.sub(" ", answer)
    answer = re.sub(r"<\|[^|>]*\|>|<[^>]+>", " ", answer)
    return " ".join(html.unescape(answer).split())


def valid(task, answer):
    if "\n" in answer or "</div>" in answer or re.search(r"</t[dhr]>", answer):
        return False
    return bool(HEADER.match(answer)) if task == "full_ocr" else not HEADER.search(answer)


def same_labels(a, b):
    A = [(m.group(1), [int(v) for v in m.groups()[1:]]) for m in HEADER.finditer(a)]
    B = [(m.group(1), [int(v) for v in m.groups()[1:]]) for m in HEADER.finditer(b)]
    def iou(p, q):
        w, h = min(p[2], q[2]) - max(p[0], q[0]), min(p[3], q[3]) - max(p[1], q[1])
        inter = max(0, w) * max(0, h)
        return inter / max(1, (p[2] - p[0]) * (p[3] - p[1]) + (q[2] - q[0]) * (q[3] - q[1]) - inter)
    return sum(any(la == lb and iou(ba, bb) >= 0.5 for lb, bb in B) for la, ba in A) / max(1, len(A))


def word_errors(answer_text, truth_text):
    a, t = Counter(answer_text.split()), Counter(truth_text.split())
    return sum((t - a).values()) + sum((a - t).values()), sum(t.values())


def clean_json(answer):
    try:
        return isinstance(json.loads(re.sub(r"\s*<\|(\d+)\|>", r" \1", answer).strip()), list)
    except ValueError:
        return False


# the same pages, in the same way, as cell 2 chose them
held_out = [row for epoch in range(1, EPOCHS + 1) for row in batch(epoch) if row["split"] == "val"]
pages = [row for task in ("full_ocr", "block_ocr") for row in [r for r in held_out if r["task"] == task][:PAGES_PER_TASK]]
real = [p for p in sorted(REAL.glob("*")) if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        and p.with_suffix(".txt").exists()] if REAL.exists() else []
jobs = ([(f"{row['folder'].name}/{row['id']}", row["task"], row) for row in pages]
        + [(f"layout/{row['folder'].name}/{row['id']}", "layout", row) for row in [r for r in held_out if r["task"] == "full_ocr"][:LAYOUT_PAGES]]
        + [(f"real/{image.name}", "real", image) for image in real])
answers = json.loads((OUT / "eval_answers.json").read_text(encoding="utf-8"))

groups, missing = {}, 0
for key, task, item in jobs:
    if f"original|{key}" not in answers or f"trained|{key}" not in answers:
        missing += 1
        continue
    o, t = answers[f"original|{key}"], answers[f"trained|{key}"]
    if task == "layout":
        g = groups.setdefault(("layout", "-"), Counter())
        g["n"] += 1; g["json_o"] += clean_json(o["answer"]); g["json_t"] += clean_json(t["answer"])
        continue
    if task == "real":
        task, lang, truth = "full_ocr", "real", " ".join(item.with_suffix(".txt").read_text(encoding="utf-8").split())
    else:
        lang, truth = item["lang"], text_of((item["folder"] / item["targets"]).read_text(encoding="utf-8"))
    g = groups.setdefault((task, lang), Counter())
    g["n"] += 1
    for side, a in (("o", o), ("t", t)):
        text = text_of(a["answer"])
        g["chars_" + side] += Levenshtein.distance(text, truth)
        e, n = word_errors(text, truth)
        g["words_" + side] += e
        g["stop_" + side] += a["stopped"]
        g["ok_" + side] += valid(task, a["answer"])
    g["chars"] += len(truth); g["words"] += n
    g["same"] += same_labels(o["answer"], t["answer"]) if task == "full_ocr" else 1

print(f"{len(jobs) - missing} of {len(jobs)} pages answered by both models" + (" (run cell 2 for the rest)" if missing else ""))
print("original -> trained (lower errors are better)")
print("  words wrong: words missed or extra, order ignored (the main score for full pages)")
print("  chars wrong: character errors in reading order (block order counts too)", end="\n\n")
for (task, lang), g in sorted(groups.items()):
    n = g["n"]
    if task == "layout":
        print(f"layout prompt ({n} pages): clean JSON {g['json_o']}/{n} -> {g['json_t']}/{n}")
        continue
    print(f"{task:9s} {lang:4s} ({n:2d} pages)  words wrong {g['words_o'] / g['words']:6.1%} -> {g['words_t'] / g['words']:6.1%}"
          f"   chars wrong {g['chars_o'] / g['chars']:6.1%} -> {g['chars_t'] / g['chars']:6.1%}"
          f"   stopped {g['stop_o']}/{n} -> {g['stop_t']}/{n}   right form {g['ok_o']}/{n} -> {g['ok_t']}/{n}"
          + (f"   same blocks as the original {g['same'] / n:.0%}" if task == "full_ocr" else ""))

45 of 45 pages answered by both models
original -> trained (lower errors are better)
  words wrong: words missed or extra, order ignored (the main score for full pages)
  chars wrong: character errors in reading order (block order counts too)

block_ocr ar   ( 7 pages)  words wrong  43.8% ->  31.2%   chars wrong   5.5% ->   2.5%   stopped 7/7 -> 7/7   right form 7/7 -> 7/7
block_ocr en   (13 pages)  words wrong  30.8% ->   0.0%   chars wrong   2.4% ->   0.0%   stopped 13/13 -> 13/13   right form 13/13 -> 13/13
full_ocr  ar   ( 6 pages)  words wrong  23.4% ->  18.6%   chars wrong  25.2% ->  15.0%   stopped 6/6 -> 6/6   right form 6/6 -> 6/6   same blocks as the original 87%
full_ocr  en   (14 pages)  words wrong  12.3% ->   1.0%   chars wrong   9.8% ->   0.6%   stopped 14/14 -> 14/14   right form 14/14 -> 14/14   same blocks as the original 75%
layout prompt (5 pages): clean JSON 4/5 -> 0/5
